# Fine‑tuning a Convolutional Neural Network with PyTorch

Fine-tuning is a powerful technique in machine learning where you take a pre-trained model and adapt it to a new, specific task. 

Normally, fine‑tune is a two-step process:
1. Load a pre-trained model
2. Replace the final layer(s) with a new layer(s) that matches the number of classes in your dataset

Fine‑tune a pre‑trained **ResNet‑18** model on the **CIFAR‑10** dataset.

### ResNet-18
Designed for ImageNet (1.28M images, 1000 classes) images, which are usually 224 x 224 pixels. It aggressively downsamples the image early on.

CIFAR-10 (50000 training images, 10 classes) images are tiny: 32 x 32 pixels.

**Solution**: Freeze the early layers of ResNet-18 and replace the final fully connected layer with 10 classes.

## 1 Setup – Imports & Device

We import the usual PyTorch libraries, set the device (GPU if available, otherwise CPU), and define a few helper functions.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Set device

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2 Load & Pre‑process CIFAR‑10

CIFAR‑10 consists of 32×32 colour images in 10 classes.  

We apply a small set of transforms – normalisation to the ImageNet statistics (because we will use a model pre‑trained on ImageNet) and conversion to tensors.

In [ ]:
# ImageNet Normalisation values used for ImageNet pre‑trained models, from Where?

imagenet_mean = [0.485, 0.456, 0.406] # mean of each channel in ImageNet dataset
imagenet_std = [0.229, 0.224, 0.225] # standard deviation of each channel in ImageNet dataset

In [ ]:
transform = transforms.Compose([ # transform the data using ImageNet mean and std
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
] )

# Download training and test sets
train_set = torchvision.datasets.CIFAR10(root='/data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.CIFAR10(root='/data', train=False, download=True, transform=transform)

# Small batch size to keep memory usage low
batch_size = 64  # Play with batch size

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
print(train_set.classes) # train_set of CIFAR10 has 10 classes

### Load ImageNet class names from URL

In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
imagenet_labels = urllib.request.urlopen(url).read().decode("utf-8").splitlines()

print("Model outputs:", len(imagenet_labels), "classes")
print(imagenet_labels[:20])   # first 20 label

## 3 Load a Pre‑trained Model & Adapt the Classifier

We will load **ResNet‑18** with ImageNet weights, freeze the early layers, and replace the final fully‑connected layer to output 10 classes.

This is a common practice when we have a large pre-trained model and want to adapt it for a smaller, domain-specific task. 

**An architecture has no knowledge about images.**

In [ ]:
# model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1) # Load ResNet‑18 pre‑trained on ImageNet
# model = models.resnet18(weights=True) # What if weights=False

# model = models.resnet18(pretrained=True) # Load ResNet‑18 pre‑trained on ImageNet
# model = models.resnet18(pretrained=False) # Load ResNet‑18 without ImageNet weights

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT) # Load ResNet18 with ImageNet weights

categories = models.ResNet18_Weights.DEFAULT.meta['categories'] # Get ImageNet categories

print(len(categories), categories)

In [ ]:
# Display model layers names

for name, detail in model.named_children(): # Important
    print(name)

In [ ]:
num_classes = model.fc.out_features
print(f"Number of output classes for ResNet18: {num_classes}")

In [ ]:
print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}\n")

# Model only
print(model)

In [ ]:
print(model.layer2) # model layer2

In [ ]:
print(model.layer2[0].conv1) # first conv layer of the second block

In [ ]:
print(model.conv1) # conv1 the first convolutional layer

#### What if **model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)**?

In [ ]:
print(model.fc) # final fully connected layer

### Freeze all layers except the final block (optional)


In [ ]:
print(model.avgpool) # average pooling layer

In [ ]:
print(model.fc)

### Freezing the Early Layers and Replace 'fc'

In [ ]:
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully‑connected layer (the classifier)
num_in_features = model.fc.in_features
num_out_features = model.fc.out_features

print(num_in_features, num_out_features) # Existing FC layer in and out features

In [ ]:
# Move model to the selected device (CPU/GPU)
model = model.to(device)

# Verify the architecture (optional)
print(model)

### 3 Customising the Classifier for Fine‑tuning

Replace the final fully‑connected layer with any architecture of yours. 

Two common patterns:
1. **Simple linear layer** – Maps the ResNet feature vector directly to the number of classes.
2. **A small head** – Adds a hidden layer, non‑linearity and dropout for better regularisation. Small head is a better choice.

Note:
Adding a head is to add a layer to the pre-trained model.

Replacing the head is to replace the layer of the pre-trained model.

In [ ]:
# Custom head with extra layers
model.fc = nn.Sequential(
    nn.Linear(num_in_features, 256),
    nn.ReLU(),
    # nn.Dropout(p=0.5),
    nn.Linear(256, 10)
)

# model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=3, bias=False)

# Unfreeze only the new head (keep earlier layers frozen)
for param in model.parameters():
    param.requires_grad = False

# For unfreezing the last layer
for param in model.fc.parameters():  # model.conv1.parameters()
    param.requires_grad = True

model = model.to(device)
print('Custom classifier attached:')
print(model.fc)

## 4 Define Loss, Optimiser & Training Loop

We use **Cross‑Entropy Loss** and the **Adam** optimiser (only for the unfrozen parameters).  
The training loop prints loss and accuracy every few batches.

In [ ]:
criterion = nn.CrossEntropyLoss() # Loss function

# Only parameters of the final layer are being optimized
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3) 

In [ ]:
def training_epoch(epoch):
    """ Train the model for one epoch """

    model.train()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_id, (inputs, targets) in enumerate(train_loader): # Training loop
        inputs, targets = inputs.to(device), targets.to(device) # move data to device

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() # sum up the loss
        _, predicted = outputs.max(1) # get the index of the max log-probability
        
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        if (batch_id + 1) % 100 == 0:
            print(f'Epoch [{epoch}] Batch [{batch_id+1}/{len(train_loader)}]  '
                  f'Loss: {running_loss/100:.4f}  Acc: {100.*correct/total:.2f}%')
            
            running_loss = 0.0
            correct = 0
            total = 0

def evaluate():
    """ Evaluate the model on the test set """

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in test_loader: # Testing loop
            inputs, targets = inputs.to(device), targets.to(device)
    
            outputs = model(inputs)
            _, predicted = outputs.max(1)

            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    acc = 100 * correct / total
    print(f'Test Accuracy: {acc:.2f}%')
    return acc

## 5 Fine‑tune the Model

Lets train for a small number of epochs.

In [ ]:
num_epochs = 5 # Total number of epochs
best_acc = 0.4 # best accuracy

for epoch in range(1, num_epochs + 1):
    training_epoch(epoch) # training the model
    acc = evaluate() # evaluate the model

    # Trying to save the best model
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'resnet18_finetuned_cifar10.pth')
        print('Saved new best model')

## Find the issue with the Accuracy and try to fix it!

## 6 Inference: Classify the provided image
Inference on the new image.

In [ ]:
# Load the best model we saved, evaluate it on test set and with the accuracy we have choosen.

model.load_state_dict(torch.load('resnet18_finetuned_cifar10.pth', map_location=device))

model.eval() # set the model to evaluation mode

### Load images & Predict

In [ ]:
from PIL import Image

class_names = train_set.classes  # list of 10 class names
print(class_names)

In [ ]:
def predict_image_class(image_path, model, class_names, device='cpu'):
    """
    Loads an image, applies necessary transformations, and predicts its class.

    Args:
        image_path (str): The path to the image file.
        model (torch.nn.Module): The pre-trained PyTorch model.
        class_names (list): A list of class names corresponding to model outputs.
        device (str): The device to run inference on ('cpu' or 'cuda').

    Returns:
        str: The predicted class name.
    """
    # Define transformations (adjust as per model's training)
    preprocess = transforms.Compose([
        # transforms.Resize(256),
        # transforms.CenterCrop(224),
        transforms.ToTensor(),
        # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    # Load and transform the image
    image = Image.open(image_path)#.convert('RGB')
    plt.imshow(image)
    plt.show()

    img_tensor = preprocess(image).unsqueeze(0).to(device) # Add batch dimension and move to device

    with torch.no_grad():
        outputs = model(img_tensor)
        _, predicted = torch.max(outputs, 1)

    print(f"Output shape: {outputs.shape}") # (Batch_Size, Num_Classes)

    probabilities = torch.nn.functional.softmax(outputs[0], dim=0)
    print(f"Top probability: {probabilities.max().item():.4f}")

    class_index = probabilities.argmax().item()
    print(f"Predicted class: {class_names[class_index]}")

    # Top 3 predictions
    pred, indices = torch.sort(outputs[0], descending=True) # remove 
    print("Three top predictions:")
    for i in range(3):
        class_index = indices[i].item()
        print(f"{class_names[class_index]}: {probabilities[class_index].item():.4f}")

    return class_names[predicted.item()]

### Load some images and collect the predicted results.

In [ ]:
images = ['images_a.jpg', 'images_b.jpg', 'images_c.jpg', 'images_d.jpg', 'images_e.jpg', 'images_f.jpg', 'images_g.jpg', 'images_h.jpg']

for img_path in images:
    predicted_class = predict_image_class(img_path, model, class_names, device)
    print(f"The predicted class for {img_path} is: {predicted_class}\n")
    print('*'*10)

# What should be done for more accuracy?



For more accuracy, we can increase the number of epochs and also use data augmentation to increase the number of images in the dataset?